# Generation of Manuscript Figures

## Overview
This notebook reproduces the primary data visualizations presented in our manuscript, *"The potential and viability of V2G for California EV drivers"*.

## Figures Generated
* **Figure 3 & Supplementary Figure 1:** Simulated 10-year capacity loss for three battery designs under the usage of the four Californian vehicle usage behavior profiles.
* **Figure 4:** Proportion of total capacity loss after 10 years attributed to calendar aging for all California vehicles
* **Figures 5, 6, & Supplementary Figure 2 - 4:** Annual V2G participation (kWh/year) and the corresponding changes in total, calendar, and cycle capacity due to V2G integration.

## Libraries and Setup

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats as st
import warnings


# Suppress minor pandas warnings for clean notebook execution
warnings.filterwarnings('ignore')

# Data 

In [ ]:
results_df = pd.read_csv('../data/california_bev_battery_degradation_simulation_results.csv')
results_df

### Vehicle Usage Behavior Clustering and Nomenclature

The results from applying clustering method, outlined in Section 3.2, to identified driver “profiles” from the real-world BEV data assigns a numeric label (`0`, `1`, `2`, `3`) to each vehicle as shown in column `cluster`. To make the context of the column `cluster`, we map the results to descriptive behavioral profiles based on the characteristics of each cluster.

**Cluster Mapping:**

| Cluster| Behavioral Profile | Key Characteristics |
| :--- | :--- | :--- |
| `0` | **Daily Charger** | *Drivers who consistently charge nightly, maintaining a high average State of Charge (SOC). * |
| `1` | **Public Charger** | *Drivers relying on public charging infrastructure, often fast charging* |
| `2` | **BEV 2nd vehicle** | *Drivers with lower utilization rates, typically charging when SOC reaches thresholds.* |
| `3` | **Threshold Charger** | *Daily users who charge only when meeting a threshold rather than every day.* |




In [ ]:

cluster_mapping = {
    0: "Daily Chargers",
    1: "Public Chargers",
    2: "BEV 2nd vehicle",
    3: "Regular Driver"
}

results_df['cluster_profile'] = results_df['cluster'].map(cluster_mapping)
results_df

# Generate Figures

In [ ]:
# Define common variables for the plots
chemistry_array = ["LFP|Gr", "NMC|Gr B1", "NMC|Gr B2"]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']

## Figure 3 (Baseline 10-Year Capacity Loss)

In [ ]:
for chemistry in chemistry_array:
    fig,ax = plt.subplots(figsize=(6,6),dpi=300)

    total_loss_df = -results_df.drop(columns = ['vehicle_id','cluster_profile']).groupby('cluster').mean()
    # Sample data
    categories = total_loss_df.index
    values1 = total_loss_df['q_loss_cal_'+chemistry+'_10_years']
    values2 = total_loss_df['q_loss_'+chemistry+'_10_years']

    x = np.arange(len(categories))  # the label locations
    x_cluster_names = np.array([cluster_mapping[val] for val in x])

    ax.bar(x_cluster_names, values1, label='Calendar Aging', color = colors, alpha =.9)
    ax.bar(x_cluster_names, values2, label='Cycle Aging', color = colors, alpha =.5)  # Stack on top of values1


    ax.set_ylabel('Capacity Change After 10 Years (%)', fontsize=16)
    ax.set_title(chemistry, fontsize=16)
    ax.set_xticks(x_cluster_names)
    ax.set_yticks([0, -5, -10, -15])#, fontsize=14)
    ax.tick_params(axis='y', labelsize=14)  


    
    ax.set_ylim([-16,0])
    fig.tight_layout(pad =1)
    

## Supplementary Figure 1 (Baseline 10-Year Capacity Loss with Error Bars)

In [ ]:
for chemistry in chemistry_array:
    fig,ax = plt.subplots(figsize=(8,5),dpi=300)

    total_loss_df = -results_df.drop(columns = ['vehicle_id','cluster_profile']).groupby('cluster').mean()
    # Sample data
    categories = total_loss_df.index
    q_loss_cal = total_loss_df['q_loss_cal_'+chemistry+'_10_years']
    q_loss_cyc = total_loss_df['q_loss_cyc_'+chemistry+'_10_years']
    q_loss = total_loss_df['q_loss_'+chemistry+'_10_years']

    # Compute the error values (standard deviation in this case)
    error_df = results_df.drop(columns = ['vehicle_id','cluster_profile']).groupby('cluster').std()  # or use .sem() for standard error
    error_total= error_df['q_loss_' + chemistry+'_10_years'] 
    error_cal = error_df['q_loss_cal_' + chemistry+'_10_years'] 
    error_cyc = error_df['q_loss_cyc_' + chemistry+'_10_years'] 


    x = np.arange(len(categories))  # the label locations



    bar_width = 0.2

    # Widths
    wide_width = 0.4
    narrow_width = 0.2
    spacing = 0.05  # controls space between bars



    ax.bar(x- 2*bar_width, q_loss,  width=2*bar_width,  label='Cycle Aging', color = colors, alpha =1)

    ax.bar(x- 2*bar_width, q_loss_cal, width=2*bar_width, label='Calendar Aging', color = colors, alpha =1)

    ax.bar(x, q_loss_cal,  width=bar_width, label='Calendar Aging', color = colors, alpha =.7)  # Stack on top 
    ax.bar(x+ bar_width*1.25, q_loss_cyc,  width=bar_width, label='Cycle Aging', color = colors, alpha =.5)  # Stack on top of 


  
    # Define error bar styling parameters
    error_kw = dict(
      ecolor='black',      # Color of the error bars
      elinewidth=1.5,        # Line width of the error bars
      capsize=5,           # Length of the error bar caps in points
      capthick=1.5           # Thickness of the error bar cap lines
    )
    # Draw error bars for the lower segment, slightly left of center
    ax.errorbar(x- 2*bar_width,  q_loss,yerr=error_total, fmt='none', **error_kw)

    # Draw error bars for the lower segment, slightly left of center
    ax.errorbar(x, q_loss_cal, yerr=error_cal, fmt='none', **error_kw)

    # Draw error bars for the upper segment, slightly right of center.
    # The y-coordinate is at the top of the second bar (i.e. values1 + values2)
    ax.errorbar(x+ bar_width*1.25, q_loss_cyc, yerr=error_cyc, fmt='none', **error_kw)



    # ax.set_xlabel('Categories')
    ax.set_ylabel('Capacity Change After 10 Years (%)', fontsize=16)
    ax.set_title(chemistry, fontsize=16)
    

    x_cluster_names = np.array([cluster_mapping[val] for val in x])

    ax.set_xticks(x)                         
    ax.set_xticklabels(x_cluster_names, fontsize=14)  
    ax.set_yticks([0, -5, -10, -15])#, fontsize=14)
    ax.tick_params(axis='y', labelsize=14)  
    # ax.set_xticklabels(categories)
    # ax.legend()
    ax.set_ylim([-16,0])

    fig.tight_layout(pad =1)
  

## Figure 4 (Proportion of total capacity loss after 10 years attributed to calendar aging for all California vehicles)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(6, 6), dpi=300, sharex=True)

for i, chemistry in enumerate(chemistry_array):
    ratio = results_df[f'q_loss_cal_{chemistry}_10_years'] / results_df[f'q_loss_{chemistry}_10_years']
    
    # Histogram
    ax[i].hist(ratio, color='grey', bins=15, edgecolor='black', label='All Vehicles')
    
    # Formatting
    ax[i].set_ylim(0, 50)
    ax[i].set_xlim(0, 1)
    ax[i].set_ylabel('Vehicle Count', fontsize=14)
    ax[i].set_title(chemistry, fontsize=16, fontweight='bold')
    
    # Vertical line for Calendar = Cycle parity
    ax[i].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Calendar Aging \n= Cycle Aging')
    ax[i].tick_params(labelsize=12)

ax[-1].set_xlabel('Calendar Aging / Total Aging', fontsize=14)
ax[0].legend(fontsize=10)
fig.tight_layout()
plt.show()

# Figure 5 and Figure 6

In [ ]:
viz_df = results_df
viz_df['V2G kwh per year'] = viz_df['V2G_kwh_10_years']/10

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
import scipy.stats as st

chemistry_array  = ["LFP|Gr", "NMC|Gr B1", "NMC|Gr B2"]


  
def confidence_interval_95(data):
    return st.t.interval(0.95, len(data)-1, loc=np.mean(data), scale=st.sem(data))


for chemistry in chemistry_array:

    sorted_df = (viz_df.drop(columns=['vehicle_id','cluster_profile'])
                 .groupby('cluster', group_keys=False)).agg(['mean', 'std', 'median',  confidence_interval_95])

    degradation_label = ['V2G_kwh_10_years', 'delta_q_loss_%s_10_years'%chemistry, 'delta_q_loss_cyc_%s_10_years'%chemistry,'delta_q_loss_cal_%s_10_years'%chemistry]

    # Create the figure with subplots
    fig = plt.figure(figsize=(9, 12), dpi=300)
    gs = gridspec.GridSpec(4, 1, height_ratios=[1, 2, 1, 1]) 
    ax = [fig.add_subplot(gs[i]) for i in range(4)]
    
    fig.suptitle(chemistry, fontsize=20)
    
    for i in range(4): 
        
        # Set the target column based on the loop index
        if i == 0: 
            data_column = 'V2G kwh per year'
        else:
            data_column = degradation_label[i]
            
        # Extract data for plotting
        averages = sorted_df[data_column]['mean'].values
        means = sorted_df[(data_column, 'mean')]
        ci_intervals = sorted_df[(data_column, 'confidence_interval_95')]

        # Correctly extract lower and upper bounds
        lower_bounds = ci_intervals.apply(lambda x: x[0])
        upper_bounds = ci_intervals.apply(lambda x: x[1])

        # Calculate error margins and combine for yerr
        lower_error = means - lower_bounds
        upper_error = upper_bounds - means
        yerr = [lower_error, upper_error]

        # Plot the bar chart
        bars = ax[i].bar([0, 1, 2, 3], averages, color=colors, width=.8, 
                         edgecolor='black', linewidth=1, yerr=yerr, capsize=5, 
                         error_kw={'elinewidth': 2, 'capthick': 2, 'ecolor': 'black'})

        # Remove default spines
        ax[i].spines['top'].set_visible(False)
        ax[i].spines['right'].set_visible(False)
        ax[i].spines['left'].set_visible(False)
        ax[i].spines['bottom'].set_visible(False)

        # Add gridlines and baseline
        ax[i].grid(axis='y', linestyle='--', alpha=0.7)
        ax[i].axhline(0, color='black', linewidth=2)  

        # Customize ticks and labels
        ax[i].tick_params(axis='y', labelsize=20)   
        ax[i].set_xticks([])
        
        if i == 0:
            ax[i].set_ylim([-10, 1500 + .3 * 1500])
            ax[i].set_ylabel('V2G\nkWh/year', fontsize=20)  
            ax[i].annotate('', xy=(0, 1500 + .3 * 1500), xytext=(0, 0),
                           xycoords=('axes fraction', 'data'), textcoords=('axes fraction', 'data'),
                           arrowprops={'arrowstyle': '->', 'color': 'black', 'linewidth': 2, 'mutation_scale': 30})

        elif i == 1:
            ax[i].set_ylim(-1.2, 1.2)
            ax[i].set_ylabel('$\\Delta$ Capacity\n(Total) (%)', fontsize=20)  
            ax[i].annotate('', xy=(0, -1.2 - 1.2 * .2), xytext=(0, 0),
                           xycoords=('axes fraction', 'data'), textcoords=('axes fraction', 'data'),
                           arrowprops={'arrowstyle': '->', 'color': 'black', 'linewidth': 2, 'mutation_scale': 30})
            ax[i].annotate('', xy=(0, 1.2 + 1.2 * .2), xytext=(0, 0),
                           xycoords=('axes fraction', 'data'), textcoords=('axes fraction', 'data'),
                           arrowprops={'arrowstyle': '->', 'color': 'black', 'linewidth': 2, 'mutation_scale': 30})
            
        elif i == 2:
            ax[i].set_ylim(-1.2, .01)
            ax[i].set_ylabel('$\\Delta$ Capacity\n(Calendar) (%)', fontsize=20)  
            ax[i].annotate('', xy=(0, -1.2 - 1.2 * .2), xytext=(0, 0),
                           xycoords=('axes fraction', 'data'), textcoords=('axes fraction', 'data'),
                           arrowprops={'arrowstyle': '->', 'color': 'black', 'linewidth': 2, 'mutation_scale': 30})

        elif i == 3:
            ax[i].set_ylim(-.01, 1.2)
            ax[i].set_ylabel('$\\Delta$ Capacity\n(Cycle) (%)', fontsize=20)
            ax[i].annotate('', xy=(0, 1.2 + 1.2 * .2), xytext=(0, 0),
                           xycoords=('axes fraction', 'data'), textcoords=('axes fraction', 'data'),
                           arrowprops={'arrowstyle': '->', 'color': 'black', 'linewidth': 2, 'mutation_scale': 30})
    
    # x-axis formatting for the final subplot
    x = np.arange(len(categories))  
    x_cluster_names = np.array([cluster_mapping[val] for val in x])

    ax[3].set_xticks(x)                         
    ax[3].set_xticklabels(x_cluster_names, fontsize=14)  
    fig.tight_layout(pad=1)
    fig.show()

# Supplementary Note 5 Figures

In [ ]:
from scipy.stats import spearmanr

from scipy.stats import linregress

## 2. Figure: Each VIN's Delta Capacity (Capacity_{with V2G} - Capacity_{no V2G}) at year 10 for all VINs

color_dict = ({0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c', 3: '#9467bd'})

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
# Example data
chemistry_array  = ["LFP|Gr", "NMC|Gr B1", "NMC|Gr B2"]


for chemistry in chemistry_array:

    sorted_df = (
        viz_df
        .sort_values(by='cluster')  # Sort by the 'cluster' row
        .groupby('cluster', group_keys=False)  # Group by 'cluster'
        .apply(lambda group: group.sort_values(by=['delta_q_loss_%s_10_years'%chemistry])))

  
  
    degradation_labels = ['delta_q_loss_%s_10_years'%chemistry, 'delta_q_loss_cyc_%s_10_years'%chemistry,'delta_q_loss_cal_%s_10_years'%chemistry]

    degradation_label_ylabel_dict ={'delta_q_loss_cyc_%s_10_years'%chemistry: "Capacity$_{\mathrm{Cycle}}$ (%)", 'delta_q_loss_cal_%s_10_years'%chemistry: "Capacity$_{\mathrm{Calendar}}$ (%)", 'delta_q_loss_%s_10_years'%chemistry : "Capacity (%)"}



    fig, ax = plt.subplots(3, 3, figsize=(12, 12),dpi=300)
    fig.suptitle(chemistry, fontweight='bold', fontsize=16)
    j =0 
    for cluster in [0,2,3]:
        cluster_sorted_df = sorted_df[sorted_df['cluster'] == cluster]
        i =0
        for degradation_label in degradation_labels: 

            ax[i,j].scatter(cluster_sorted_df ['V2G_kwh_10_years']/10,cluster_sorted_df [degradation_label],  color = color_dict[cluster])
            ax[i,j].set_xlabel(r'$V2G \ \frac{\mathrm{kWh}}{\mathrm{year} \cdot \mathrm{person}}$', fontsize=14)
            ax[i,j].set_ylabel(r'$\Delta$ %s'%degradation_label_ylabel_dict[degradation_label], fontsize=14)

            slope, intercept, r_value, p_value, std_err = linregress(cluster_sorted_df ['V2G_kwh_10_years']/10,cluster_sorted_df [degradation_label])

            
            x_max = cluster_sorted_df['V2G_kwh_10_years'].max() / 10
            x_range = np.linspace(0, x_max, 100)
            y_line = slope * x_range
            
            ax[i,j].plot(x_range, y_line, 'r--', linewidth=2)

            # correlation, p_value = spearmanr(cluster_sorted_df ['V2G kwh']/10,-cluster_sorted_df [degradation_label])

            ax[i,j].set_title('slope: %.2f e-4 \n r-value: %f \n p-value: %.2f \n std_err: %.2f e-4'%( slope*10000, r_value, p_value, std_err*10000 ), fontsize=12)



            ax[i,j].set_xlim([-10, 1300])
            i = i+1



        ax[0,j].set_ylim(-2, 2)

        ax[1,j].set_ylim(-2, .1)

        ax[2,j].set_ylim(-.1, 2)
        j= j+1

          # Show the plot
        fig.tight_layout(pad=1)
        fig.show()


